In [11]:
import os
import cv2
import glob
import numpy as np
import torch
import random
import hashlib
import gc
import torch.nn as nn
import torch.optim as optim
import albumentations as A
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import VideoMAEForVideoClassification, VideoMAEImageProcessor
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from collections import Counter
from imblearn.over_sampling import RandomOverSampler

# Load Pretrained VideoMAE Model & Processor
model_name = "MCG-NJU/videomae-base"
base_model = VideoMAEForVideoClassification.from_pretrained(model_name, num_labels=2, ignore_mismatched_sizes=True)
processor = VideoMAEImageProcessor.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Modify Model to Add Dropout
class ModifiedVideoMAE(nn.Module):
    def __init__(self, base_model, num_labels=2, dropout_rate=0.5):
        super(ModifiedVideoMAE, self).__init__()
        self.videomae = base_model
        self.dropout = nn.Dropout(dropout_rate)
        # سنستخدم hidden_size من إعدادات النموذج كأساس لطبقة التصنيف المخصصة
        self.classifier = nn.Linear(self.videomae.config.hidden_size, num_labels)
        
    def forward(self, pixel_values):
        # نفترض أن بإمكاننا الحصول على المميزات من النموذج عبر output_hidden_states
        outputs = self.videomae(pixel_values=pixel_values, output_hidden_states=True)
        # نستخدم المميزة المقابلة للرمز الأول أو المتوسط (يجب التأكد من بنية VideoMAE)
        # هنا مثال باستخدام الرمز الأول من آخر طبقة
        features = outputs.hidden_states[-1][:, 0]
        features = self.dropout(features)
        return self.classifier(features)



# Initialize Model
model = ModifiedVideoMAE(base_model)
model.to(device)

# Dataset Path
DATASET_PATH = "Shop_DataSet"
CATEGORIES = ["non_shop_lifters", "shop_lifters"]

# Augmentations
augmentations = A.Compose([
    A.RandomResizedCrop(size=(224, 224), scale=(0.6, 1.0), ratio=(0.75, 1.33), p=0.5),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.MotionBlur(blur_limit=(3, 5), p=0.2),
    A.ShiftScaleRotate(shift_limit=0.03, scale_limit=0.03, rotate_limit=15, p=0.4),
    A.GaussNoise(var_limit=(15.0, 40.0), p=0.3),
    A.CLAHE(clip_limit=2.0, p=0.3),
    A.Normalize(mean=(0.5,), std=(0.5,)),
    ToTensorV2(),
])


# Extract Frames using Adaptive Sampling
def extract_frames(video_path, num_frames=16):
    cap = cv2.VideoCapture(video_path)
    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if total_frames == 0:
        cap.release()
        return None
    
    frame_indices = np.linspace(0, total_frames - 1, num_frames * 2, dtype=int)
    sampled_frames = []
    
    prev_frame = None
    for i in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()
        if not ret:
            continue
        
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (224, 224))
        
        if prev_frame is None or np.mean(np.abs(frame - prev_frame)) > 15:
            sampled_frames.append(frame)
            prev_frame = frame
        
        if len(sampled_frames) == num_frames:
            break
    
    cap.release()
    return sampled_frames if len(sampled_frames) == num_frames else None

# Load Videos and Balance Dataset
video_paths, labels = [], []
for label, category in enumerate(CATEGORIES):
    video_folder = os.path.join(DATASET_PATH, category)
    video_files = glob.glob(os.path.join(video_folder, "*.mp4"))

    for video_file in video_files:
        frames = extract_frames(video_file)
        if frames is not None:
            video_paths.append(video_file)
            labels.append(label)

# Balance Dataset
labels_array = np.array(labels).reshape(-1, 1)
video_paths_array = np.array(video_paths)

oversampler = RandomOverSampler()
video_paths_resampled, labels_resampled = oversampler.fit_resample(video_paths_array.reshape(-1, 1), labels_array)

video_paths = video_paths_resampled.flatten().tolist()
labels = labels_resampled.flatten().tolist()

print(f"New Balanced Dataset: {Counter(labels)}")

# عدد الفريمات المستخرجة من كل فيديو
num_frames = 16

# تعريف الكلاس الخاص بالـ Dataset
class ShopliftingDataset(Dataset):
    def __init__(self, video_paths, labels, processor, num_frames=16):
        self.video_paths = video_paths
        self.labels = labels
        self.processor = processor
        self.num_frames = num_frames  # تخزين عدد الفريمات في الكائن

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        frames = extract_frames(video_path, num_frames=self.num_frames)  # تمرير num_frames هنا

        if frames is None:
            return None
        
        inputs = self.processor(frames, return_tensors="pt")
        return inputs["pixel_values"].squeeze(0), torch.tensor(label, dtype=torch.long)

# Split Dataset
dataset = ShopliftingDataset(video_paths, labels, processor, num_frames=num_frames)
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

# Define Loss, Optimizer, and Scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2, verbose=True)

# Train Model with Early Stopping
early_stopping_patience = 4
best_val_loss = float("inf")
epochs_no_improve = 0
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0

    for inputs, labels in tqdm(train_loader):
        gc.collect()
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(pixel_values=inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    scheduler.step(avg_loss)

    if avg_loss < best_val_loss:
        best_val_loss = avg_loss
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= early_stopping_patience:
        print("Early stopping triggered!")
        break

# Evaluate Model
def evaluate_model(model, test_loader):
    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(pixel_values=inputs)
            preds = torch.argmax(outputs, dim=1)
            test_preds.extend(preds.cpu().numpy())
            test_labels.extend(labels.cpu().numpy())

    print(f"Test Accuracy: {accuracy_score(test_labels, test_preds):.4f}")

evaluate_model(model, test_loader)


Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Mohamed\AppData\Roaming\Python\Python310\site-packages\albumentations\core\validation.py:87: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
C:\Users\Mohamed\AppData\Local\Temp\ipykernel_31776\3194874101.py:63: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(15.0, 40.0), p=0.3),
c:\Users\Mohamed\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


New Balanced Dataset: Counter({0: 531, 1: 531})


100%|██████████| 186/186 [1:46:54<00:00, 34.48s/it]


Test Accuracy: 0.9500


In [12]:
torch.save(model.state_dict(), "shoplifting_model.pth")
print("Model saved successfully.")


Model saved successfully.


In [18]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Function to evaluate model on any DataLoader
def evaluate_loader(model, data_loader, criterion, device, name="Set"):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in data_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(pixel_values=inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    cm = confusion_matrix(all_labels, all_preds)

    print(f"\n--- {name} Evaluation ---")
    print(f"Loss: {avg_loss:.4f}")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Confusion Matrix:\n{cm}")

    return {
        "loss": avg_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm
    }

# Example usage:
train_results = evaluate_loader(model, train_loader, criterion, device, name="Train")
val_results = evaluate_loader(model, val_loader, criterion, device, name="Validation")
test_results = evaluate_loader(model, test_loader, criterion, device, name="Test")


--- Train Evaluation ---
Loss: 0.0552
Accuracy: 0.9838
Precision: 0.9694
Recall: 1.0000
F1 Score: 0.9845
Confusion Matrix:
[[351  12]
 [  0 380]]

--- Validation Evaluation ---
Loss: 0.1356
Accuracy: 0.9497
Precision: 0.9000
Recall: 1.0000
F1 Score: 0.9474
Confusion Matrix:
[[79  8]
 [ 0 72]]

--- Test Evaluation ---
Loss: 0.0925
Accuracy: 0.9500
Precision: 0.9080
Recall: 1.0000
F1 Score: 0.9518
Confusion Matrix:
[[73  8]
 [ 0 79]]


In [22]:
video_path = "Shop_DataSet/non_shop_lifters/shop_lifter_n_76_1.mp4"


In [23]:
def predict_video(model, video_path, processor, num_frames=16):
    model.eval()
    frames = extract_frames(video_path, num_frames=num_frames)
    if frames is None:
        print("❌ Failed to extract frames.")
        return

    inputs = processor(frames, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)
    
    with torch.no_grad():
        outputs = model(pixel_values=pixel_values)
        probs = torch.softmax(outputs, dim=1)
        predicted_class = torch.argmax(probs, dim=1).item()
        confidence = probs[0, predicted_class].item()

    label = CATEGORIES[predicted_class]
    print(f"\n✅ Prediction: {label}")
    print(f"🎯 Confidence: {confidence * 100:.2f}%")

# مثال:
predict_video(model, "Shop_DataSet/non_shop_lifters/shop_lifter_n_76_1.mp4", processor)



✅ Prediction: non_shop_lifters
🎯 Confidence: 100.00%


In [ ]:
# تعريف نفس الكلاس
model_name = "MCG-NJU/videomae-base"
base_model = VideoMAEForVideoClassification.from_pretrained(model_name, num_labels=2, ignore_mismatched_sizes=True)
loaded_model = ModifiedVideoMAE(base_model)
loaded_model.load_state_dict(torch.load("shoplifting_model.pth"))
loaded_model.to(device)
loaded_model.eval()
print("Model loaded successfully.")


Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


FileNotFoundError: [Errno 2] No such file or directory: 'shop_non_or_lifting_model.pth'